# JIT 编译流程与离线二进制

第 1 章说过 Pro 具有"编译器属性"。本节把编译的细节讲清：JIT 的完整流程、**编译签名**（决定何时重新编译）、以及面向交付的离线二进制形态。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. JIT 编译流程回顾

Kernel 首次通过 `kernel[stream, block_dim](...)` 启动时触发 JIT 编译：

1. **绑定**：绑定实参与启动配置，确定动态参数与编译期特化信息；
2. **解析**：解析 Kernel 函数体，生成 PyPTO IR 并校验；
3. **转换**：对 IR 进行必要转换，生成 Device 代码和 Host 侧 Launcher；
4. **编译加载**：编译并加载产物，向指定 Stream 下发 Kernel。

Kernel 启动相对 Host 异步，但首次调用会先等待当前编译实例生成完成。


## 2. 编译签名：什么触发重新编译

同一 Kernel 对象按**编译签名**区分编译实例：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">信息</th><th style="text-align:left;">对编译实例的影响</th></tr>
<tr><td>Tensor 固定维度</td><td>编入 IR，调用时必须匹配</td></tr>
<tr><td>STATIC 维度</td><td>运行时取值参与特化，值变化生成新实例</td></tr>
<tr><td>DYNAMIC 维度</td><td>不参与特化，值变化复用实例</td></tr>
<tr><td>Scalar 入参（DT_*）</td><td>类型由标注确定；<b>值变化不产生新实例</b></td></tr>
<tr><td>TilingKey</td><td>每个合法 Key 对应专用实例</td></tr>
<tr><td>datatype</td><td>每组类型组合对应专用实例</td></tr>
<tr><td>编译目标（arch）</td><td>不同目标使用不同 Kernel 对象</td></tr>
</table>

两个容易踩的点：

- **`stream` 和 `block_dim` 不参与编译签名**——调整两者不会产生新编译实例；
- **JIT 复用范围限于当前 Python 进程**：重启进程会重新编译；`build` 目录用于加载与调试，不是跨进程持久缓存。


## 3. 实验设计：观察编译与复用


In [ ]:
import time

@pl.jit(auto_mutex=True)
def dynamic_add(
    x: pl.Tensor[[pl.DYNAMIC, 256], pl.DT_FP32],
    y: pl.Tensor[[pl.DYNAMIC, 256], pl.DT_FP32],
    out: pl.Tensor[[pl.DYNAMIC, 256], pl.DT_FP32],
):
    tt = pl.TileType(shape=[1, 256], dtype=pl.DT_FP32, target_memory=pl.MemorySpace.Vec)
    gx = pl.make_tile_group(type=tt, addrs=0x0000, mutex_ids=[0])
    gy = pl.make_tile_group(type=tt, addrs=0x0400, mutex_ids=[1])
    go = pl.make_tile_group(type=tt, addrs=0x0800, mutex_ids=[2])
    with pl.section_vector():
        xt, yt, ot = gx.current(), gy.current(), go.current()
        for row in pl.range(x.shape[0]):
            pl.load(xt, x, [row, 0])
            pl.load(yt, y, [row, 0])
            pl.add(ot, xt, yt)
            pl.store(out, ot, [row, 0])

device = f"npu:{os.environ['TILE_FWK_DEVICE_ID']}"
torch.npu.set_device(device)
x1 = torch.randn(16, 256, device=device)
y1 = torch.randn_like(x1)
out1 = torch.empty_like(x1)
x2 = torch.randn(32, 256, device=device)
y2 = torch.randn_like(x2)
out2 = torch.empty_like(x2)
durations = []
for x, y, out in [(x1, y1, out1), (x1, y1, out1), (x2, y2, out2)]:
    torch.npu.synchronize()
    start = time.perf_counter()
    dynamic_add[None, 1](x, y, out)
    torch.npu.synchronize()
    durations.append(time.perf_counter() - start)
    torch.testing.assert_close(out, x + y, rtol=1e-5, atol=1e-6)
print('首次、同形状复用、动态形状复用的 Host 总耗时（秒）：', durations)

【待 950 验证】首次调用含编译开销；后两次按相同动态签名复用。形状变大意味着工作量变化，不能据“复用实例”推断耗时相同。本实验记录 Host 总耗时，不是纯 Device Kernel 耗时；确认编译次数还需结合日志/编译产物。


### 给三次调用分别命名，才看得懂时间

一次实验记录至少包含输入 shape/dtype、静态配置、启动核数、是否首次调用、同步位置与墙钟耗时。同签名第二次调用用于观察复用，另一个动态行数用于观察循环覆盖；仅看到一次调用更快，不能判断缓存命中或设备性能提升。

例如第一次为 128 行、第二次同为 128 行、第三次为 256 行。若行数是动态维度，前两次的工作量相同，第三次却翻倍。即使第三次复用编译实例，执行时间也可能增长。若改变的是编译期 Tile 容量，又是另一类实验。不要把“编译复用”和“工作量不变”合成同一个结论。


## 4. 离线二进制编译：交付形态

JIT 适合开发调试；**算子随包发布**时用离线编译，生成随算子包安装的 AI Core 二进制，调用时不触发即时编译。主要步骤：

1. 准备算子定义、InferShape、Host 侧 Tiling 与调用接口；
2. 在 `op_kernel` 目录实现 PyPTO Pro Kernel；
3. 在 `op_host/CMakeLists.txt` 中启用该 Kernel；
4. Host 侧填写 TilingData、TilingKey、BlockDim 与 Workspace；
5. 运行算子工程构建脚本，生成算子安装包。

离线 Kernel 的结构要求（与 JIT 写法的关键差异）：

- 每个 `op_kernel/*.py` 恰好定义**一个** `@pl.jit` Kernel；
- 定义 TilingKey 并通过 `tiling_key` 参数绑定；
- TilingData 用 dataclass 定义，**恰好一个**且位于参数列表**末尾**；
- Workspace 参数位于业务输入输出之后、TilingData 之前（参数结尾形如 `..., workspace, tiling`）；
- Kernel 函数名与算子入口名称一致，业务参数名与算子原型一致。

```text
# 离线工程结构示意，省略业务参数与类型特化配置，不可直接执行
from dataclasses import dataclass
import pypto_pro.language as pl
from pypto_pro.runtime.tilingkey import TilingKeyField


@dataclass
class AddExampleTilingData:
    total_length: int
    tile_num: int


class AddExampleTilingKey:
    sch_mode = TilingKeyField(bits=1, values=[0, 1])


@pl.jit(tiling_key=AddExampleTilingKey, datatype={...})
def add_example(..., workspace, tiling):
    ...
```

【待 950 验证：结构取自官方离线编译文档；本课程以 JIT 为主线，离线交付在需要时按文档落地】


## 5. 课后练习

1. （填空）Kernel 两次调用之间只改了 `block_dim`（1 核 → 4 核），会生成新的编译实例吗？______。只改了某个 `pl.STATIC` 维度的值呢？______。
2. （判断）JIT 编译结果保存在 build 目录，重启 Python 进程后可直接复用。（对/错）
3. （选择）离线二进制 Kernel 的 TilingData 参数位置要求：
   A. 任意位置<br/>
   B. 参数列表末尾，且全 Kernel 恰好一个<br/>
   C. 与 workspace 之前即可，可有多个

In [ ]:
!cat ./answer/02.07_answer.txt

## 6. 本节小结

- JIT 四步：绑定 → 解析（PyPTO IR）→ 转换（Device 代码 + Launcher）→ 编译加载；
- 编译签名看**维度声明方式**（固定/STATIC 特化、DYNAMIC 复用）、TilingKey、datatype；stream/block_dim 与标量**值**不参与；
- 复用限于进程内；离线二进制面向随包交付，TilingData 必须收尾且唯一。

至此第 2 章概念全部讲完。下一节综合实践。